# 🐬 Dolphin Platform V2 — Demonstration Notebook

**AI-Augmented Chemical Research Management**

---

## Abstract

This notebook provides an interactive walkthrough of the Dolphin Platform's core cheminformatics
and AI capabilities. We demonstrate:

1. **SMILES parsing & molecular visualization** via RDKit
2. **Functional group detection** using SMARTS substructure matching
3. **Molecular descriptor computation** for drug-likeness assessment

> **Target Audience**: Researchers at the intersection of machine learning and chemical sciences.

---

## 1. Setup & Imports

In [ ]:
import sys
import os

# Add project root to path for imports
sys.path.insert(0, os.path.abspath('..'))

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, rdMolDraw2D
from IPython.display import SVG, display, HTML
import pandas as pd

from utils.chem_utils import compute_functional_groups, generate_structure_svg

print(f"RDKit version: {Chem.rdBase.rdkitVersion}")
print("✅ All imports successful")

## 2. Chemical Data Handling — SMILES Parsing

SMILES (Simplified Molecular Input Line Entry System) is the canonical string representation
used throughout Dolphin Platform. We demonstrate parsing and canonical form generation.

In [ ]:
# Define a set of pharmaceutically relevant molecules
molecules = {
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Caffeine": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "Penicillin G": "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",
}

# Parse and canonicalize
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    canonical = Chem.MolToSmiles(mol, canonical=True)
    inchi = Chem.MolToInchi(mol)
    print(f"{name:15s} | SMILES: {canonical}")
    print(f"{'':15s} | InChI:  {inchi[:60]}...")
    print()

## 3. Molecular Visualization

Dolphin Platform uses RDKit's `MolDraw2DSVG` renderer for publication-quality 2D structures.

In [ ]:
# Render a grid of molecules
mols = [Chem.MolFromSmiles(smi) for smi in molecules.values()]
legends = list(molecules.keys())

img = Draw.MolsToGridImage(
    mols, 
    molsPerRow=3, 
    subImgSize=(350, 300),
    legends=legends
)
display(img)

In [ ]:
# SVG rendering using Dolphin's utility
svg = generate_structure_svg("CC(=O)Oc1ccccc1C(=O)O", width=400, height=350)
display(SVG(svg))
print("Aspirin — rendered via utils.chem_utils.generate_structure_svg")

## 4. Functional Group Detection

The platform detects functional groups via SMARTS substructure matching — a pattern language
for molecular substructures. This powers the chemical inventory's automatic classification.

In [ ]:
# Detect functional groups for all molecules
results = []
for name, smi in molecules.items():
    groups = compute_functional_groups(smi)
    results.append({
        "Molecule": name,
        "SMILES": smi,
        "Functional Groups": ", ".join(groups),
        "Count": len(groups),
    })

df = pd.DataFrame(results)
display(df.style.set_caption("Functional Group Detection via SMARTS Matching"))

## 5. Molecular Descriptors & Drug-Likeness

We compute Lipinski's Rule of Five descriptors — a key filter in drug discovery pipelines.

In [ ]:
# Compute descriptors
descriptor_data = []
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    descriptor_data.append({
        "Molecule": name,
        "MW (Da)": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "TPSA (Å²)": round(Descriptors.TPSA(mol), 1),
        "Lipinski Pass": "✅" if (
            Descriptors.MolWt(mol) <= 500 and
            Descriptors.MolLogP(mol) <= 5 and
            Descriptors.NumHDonors(mol) <= 5 and
            Descriptors.NumHAcceptors(mol) <= 10
        ) else "❌",
    })

df_desc = pd.DataFrame(descriptor_data)
display(df_desc.style.set_caption("Lipinski Rule of Five — Drug-Likeness Assessment"))

## 6. Results & Discussion

### Key Findings

| Capability | Implementation | Status |
|------------|---------------|--------|
| SMILES Parsing | RDKit `MolFromSmiles` | ✅ Production |
| Functional Group Detection | 45+ SMARTS patterns | ✅ Production |
| SVG Rendering | `MolDraw2DSVG` | ✅ Production |
| Lipinski Filtering | `Descriptors` module | ✅ Production |
| OCSR (MolScribe) | PPM Pipeline | ✅ Production |
| AI Plate Layout | Screenings module | ✅ Production |

### Architecture Highlights

- **Modular blueprint design**: Each analytical domain (LC-MS, PPM, Screenings) is a self-contained Flask blueprint
- **AI as infrastructure**: LLM capabilities are embedded in the workflow, not bolted on afterwards
- **Closed-loop system**: From PDF intake → OCSR → SMILES → inventory → plate design → LC-MS analysis

### References

1. Schwaller, P. et al. (2019). *Molecular Transformer*. ACS Cent. Sci.
2. Boiko, D. et al. (2023). *Autonomous chemical research with large language models*. Nature.
3. Xu, Z. et al. (2022). *MolScribe: Robust Molecular Structure Recognition*.
4. Weininger, D. (1988). *SMILES: A Chemical Language*. J. Chem. Inf. Comput. Sci.

---

*Dolphin Platform V2 — Built for accelerated chemical discovery.*